In [1]:
import re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 200)

def norm_col(name: str) -> str:
    """Maak kolomnamen klein en 'snake_case'."""
    name = str(name).strip().lower()
    name = re.sub(r"\s+", "_", name)
    name = name.replace("-", "_")
    return name

In [2]:
# Pad naar je eigen dataset (pas aan indien nodig)
DATA_PATH = Path("Data/vkm-dataset.csv")

df_raw = pd.read_csv(DATA_PATH, sep=None, engine="python")  # autodetect ; of ,
df = df_raw.copy()

# Kolomnamen normaliseren
df.columns = [norm_col(c) for c in df.columns]

print("Vorm van de data:", df.shape)
display(df.head())
print("\nKolominformatie:")
print(df.info())
print("\nAantal missende waarden per kolom:")
print(df.isna().sum())


Vorm van de data: (211, 20)


,id,name,shortdescription,description,content,studycredit,location,contact_id,level,learningoutcomes,rood,groen,blauw,geel,module_tags,interests_match_score,popularity_score,estimated_difficulty,available_spots,start_date
0,159,Kennismaking met Psychologie,"Brein, gedragsbeinvloeding, ontwikkelingspsychologie, gespreksvoering en ontwikkelingsfasen.",In deze module leer je hoe je gedrag van jezelf en van anderen kunt begrijpen en beinvloeden. Je maakt kennis met de basistheorie van Psychologie. Aan bod komen onderwerpen die te maken hebben met...,In deze module leer je hoe je gedrag van jezelf en van anderen kunt begrijpen en beinvloeden. Je maakt kennis met de basistheorie van Psychologie. Aan bod komen onderwerpen die te maken hebben met...,15,Den Bosch,58,NLQF5,A. Je beantwoordt vragen in een meerkeuze kennistoets waarin je laat zien dat je de basis van de psychologie kunt reproduceren en begrijpt. Je laat zien dat je gedrag van individuen en groepen in ...,4.0,2.0,1.0,5.0,"['brein', 'gedragsbeinvloeding', 'ontwikkelingspsychologie', 'gespreksvoering', 'en', 'ontwikkelingsfasen']",0.54,319,1,79,2025-12-24
1,160,Learning and working abroad,"Internationaal, persoonlijke ontwikkeling, verpleegkunde","Studenten kiezen binnen de (stam) van de opleiding van Verpleegkunde steeds vaker voor een stage in het buitenland, waarbij zij de beroepsprestaties graag in een internationale stagecontext willen...","Studenten kiezen binnen de (stam) van de opleiding van Verpleegkunde steeds vaker voor een stage in het buitenland, waarbij zij de beroepsprestaties graag in een internationale stagecontext willen...",15,Den Bosch,58,NLQF5,De student toont professioneel gedrag conform de beroepscode bij laagcomplexe zorgvragers en collega's in de zorgsetting.,5.0,3.0,1.0,1.0,"['internationaal', 'persoonlijke', 'ontwikkeling', 'verpleegkunde']",0.92,172,5,56,2025-12-20
2,161,Proactieve zorgplanning,"Proactieve zorgplanning, cocreatie, ziekenhuis","Het Jeroen Bosch ziekenhuis wil graag samen met de opleiding Verpleegkunde een module ontwikkelen, waarin de studenten de mogelijkheid krijgen om zich te verdiepen in de ziekenhuissetting. JBZ sta...","Het Jeroen Bosch ziekenhuis wil graag samen met de opleiding Verpleegkunde een module ontwikkelen, waarin de studenten de mogelijkheid krijgen om zich te verdiepen in de ziekenhuissetting. JBZ sta...",15,Den Bosch,59,NLQF5,"De student past pro actieve zorgplanning toe binnen het proces van het klinisch redeneren bij casuïstiek over zorgvragers, multidisciplinair, in de ziekenhuissetting.",NaN,NaN,NaN,NaN,"['proactieve', 'zorgplanning', 'cocreatie', 'ziekenhuis']",0.78,217,5,55,2025-09-23
3,162,Rouw en verlies,"Rouw & verlies, palliatieve zorg & redeneren, trauma, culturele diversiteit","In deze module wordt stil gestaan bij rouw en verlies, vanuit diverse invalshoeken waaronder de palliatieve zorg. Thema's zoals oncologie kunnen hier een plaats krijgen (werkveld verpleegkunde vra...","In deze module wordt stil gestaan bij rouw en verlies, vanuit diverse invalshoeken waaronder de palliatieve zorg. Thema's zoals oncologie kunnen hier een plaats krijgen (werkveld verpleegkunde vra...",30,Den Bosch,58,NLQF6,De student regisseert en voert (deels) zelfstandig en in samenwerkingsverband het palliatieve proces uit bij hoog/middencomplexe zorgvragers in de zorgsetting op basis van het proces van palliatie...,NaN,NaN,NaN,NaN,"['rouw', 'verlies', 'palliatieve', 'zorg', 'redeneren', 'trauma', 'culturele', 'diversiteit']",0.69,454,1,54,2025-10-25
4,163,Acuut complexe zorg,"Acute zorg, complexiteit, ziekenhuis, revalidatie","In deze module kunnen studenten zich verdiepen in de acuut, complexe zorg binnen het verpleegkundig vakgebied.","In deze module kunnen studenten zich verdiepen in de acuut, complexe zorg binnen het verpleegkundig vakgebied.",30,Den Bosch,58,NLQF6,De student regisseert en voert (deels) zelfstandig en in samenwerkingsverband het verpleegkundige proces uit bij hoog/middencomplexe zorgvragers in de actue en


Kolominformatie:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 211 entries, 0 to 210
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     211 non-null    int64  
 1   name                   211 non-null    object 
 2   shortdescription       191 non-null    object 
 3   description            211 non-null    object 
 4   content                211 non-null    object 
 5   studycredit            211 non-null    int64  
 6   location               211 non-null    object 
 7   contact_id             211 non-null    int64  
 8   level                  211 non-null    object 
 9   learningoutcomes       206 non-null    object 
 10  rood                   2 non-null      float64
 11  groen                  2 non-null      float64
 12  blauw                  2 non-null      float64
 13  geel                   2 non-null      float64
 14  module_tags            211 non-null    o

In [3]:
ALIASES: Dict[str, List[str]] = {
    "module_id": ["module_id", "id"],
    "title": ["module_title", "title", "name"],
    "short_description": ["short_description", "shortdescription"],
    "description": ["module_descripiton", "description", "omschrijving"],
    "content": ["content", "inhoud"],
    "study_credits": ["study_credits", "studycredits", "studycredit"],
    "location": ["location", "locatie", "standplaats"],
    "contact_id": ["contact_id"],
    "level": ["level"],
    "learning_outcomes": ["learning_outcomes", "learningoutcomes"],
    "red": ["red", "rood"],
    "green": ["green", "groen"],
    "blue": ["blue", "blauw"],
    "yellow": ["yellow", "geel"],
    "module_tags": ["module_tags", "tags"],
    "interests_match_score": ["interests_match_score"],
    "popularity_score": ["popularity_score"],
    "estimated_difficulty": ["estimated_difficulty"],
    "available_spots": ["available_spots"],
    "start_date": ["start_date"],
}

def pick_col(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    cols = list(df.columns)
    for alias in aliases:
        alias_norm = norm_col(alias)
        for c in cols:
            if norm_col(c) == alias_norm:
                return c
    return None

COLS: Dict[str, Optional[str]] = {key: pick_col(df, aliases) for key, aliases in ALIASES.items()}
COLS


{'module_id': 'id',
 'title': 'name',
 'short_description': 'shortdescription',
 'description': 'description',
 'content': 'content',
 'study_credits': 'studycredit',
 'location': 'location',
 'contact_id': 'contact_id',
 'level': 'level',
 'learning_outcomes': 'learningoutcomes',
 'red': 'rood',
 'green': 'groen',
 'blue': 'blauw',
 'yellow': 'geel',
 'module_tags': 'module_tags',
 'interests_match_score': 'interests_match_score',
 'popularity_score': 'popularity_score',
 'estimated_difficulty': 'estimated_difficulty',
 'available_spots': 'available_spots',
 'start_date': 'start_date'}

In [4]:
text_parts = []

for key in ["shortdescription", "description", "content", "learningoutcomes", "module_tags"]:
    col = COLS.get(key)
    if col and col in df.columns:
        text_parts.append(df[col].fillna("").astype(str))

if not text_parts:
    raise ValueError("Geen geschikte tekstkolommen gevonden om 'module_text' te maken. Controleer je data.")

module_text = text_parts[0]
for part in text_parts[1:]:
    module_text = module_text + " " + part

df["module_text"] = module_text.str.replace(r"\s+", " ", regex=True).str.strip()
df[[COLS.get("title") or df.columns[0], "module_text"]].head()


,name,module_text
0,Kennismaking met Psychologie,In deze module leer je hoe je gedrag van jezelf en van anderen kunt begrijpen en beinvloeden. Je maakt kennis met de basistheorie van Psychologie. Aan bod komen onderwerpen die te maken hebben met...
1,Learning and working abroad,"Studenten kiezen binnen de (stam) van de opleiding van Verpleegkunde steeds vaker voor een stage in het buitenland, waarbij zij de beroepsprestaties graag in een internationale stagecontext willen..."
2,Proactieve zorgplanning,"Het Jeroen Bosch ziekenhuis wil graag samen met de opleiding Verpleegkunde een module ontwikkelen, waarin de studenten de mogelijkheid krijgen om zich te verdiepen in de ziekenhuissetting. JBZ sta..."
3,Rouw en verlies,"In deze module wordt stil gestaan bij rouw en verlies, vanuit diverse invalshoeken waaronder de palliatieve zorg. Thema's zoals oncologie kunnen hier een plaats krijgen (werkveld verpleegkunde vra..."
4,Acuut complexe zorg,"In deze module kunnen studenten zich verdiepen in de acuut, complexe zorg binnen het verpleegkundig vakgebied. In deze module kunnen studenten zich verdiepen in de acuut, complexe zorg binnen het ..."


In [5]:
# nltk.download('stopwords')
dutch_stop_words = stopwords.words('dutch')

tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    stop_words=dutch_stop_words  # gebruik 'english' voor Engelstalige data
)

X = tfidf.fit_transform(df["module_text"].fillna(""))
X.shape


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gebruiker\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


(211, 8000)

In [8]:
@dataclass
class StudentProfile:
    interests_text: str
    study_credits: Optional[int] = None
    preferred_location: Optional[str] = None
    preferred_level: Optional[str] = None
    max_difficulty: Optional[float] = None

def profile_vector(profile: CandidateProfile):
    return tfidf.transform([profile.interests_text])


In [9]:
def extract_numeric_series(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.replace(",", ".", regex=False)
    nums = pd.to_numeric(
        s.str.extract(r"([0-9]+\.?[0-9]*)")[0],
        errors="coerce"
    )
    return nums

def constraint_score(row: pd.Series, profile: StudentProfile) -> Tuple[float, Dict[str, str]]:
    """
    Berekent een gemiddelde score (0.0 tot 1.0) op basis van harde en zachte constraints
    van het studentprofiel.
    """
    scores = []
    reasons: Dict[str, str] = {}

    ## 1. Locatie Constraint
    loc_col = COLS.get("location")
    if profile.preferred_location and loc_col and loc_col in row.index:
        module_loc = str(row[loc_col]).lower()
        pref = profile.preferred_location.lower()
        # Controleert of de voorkeurslocatie in de module-locatie staat
        match = pref in module_loc
        scores.append(1.0 if match else 0.0)
        reasons["location"] = f"Locatie {'matcht' if match else 'matcht niet'}: student={profile.preferred_location}, module={row[loc_col]}"
    else:
        reasons["location"] = "Geen locatie-voorkeur opgegeven."

    ## 2. Study Credits Constraint (Nieuw)
    credit_col = COLS.get("study_credits")
    if profile.study_credits is not None and credit_col and credit_col in row.index:
        # We nemen aan dat study_credit al numeriek is, maar extract het voor de zekerheid
        module_credits = extract_numeric_series(pd.Series([row[credit_col]])).iloc[0]
        
        if pd.isna(module_credits):
             scores.append(0.5)
             reasons["study_credits"] = "Studiepunten onbekend; neutrale score."
        else:
             # De constraint is: module moet exact dit aantal credits hebben (of een range,
             # maar voor nu doen we een exacte match voor eenvoud)
             match = module_credits == profile.study_credits
             scores.append(1.0 if match else 0.0)
             reasons["study_credits"] = f"Studiepunten {'match' if match else 'geen match'}: gewenst={profile.study_credits}, module={module_credits}."
    else:
        reasons["study_credits"] = "Geen studiepunten-constraint of kolom."
        
    ## 3. Preferred Level Constraint (Nieuw)
    level_col = COLS.get("level")
    if profile.preferred_level and level_col and level_col in row.index:
        module_level = str(row[level_col]).lower()
        pref = profile.preferred_level.lower()
        
        # We zoeken naar een exacte of gedeeltelijke match (bijv. NLQF6)
        match = pref in module_level
        scores.append(1.0 if match else 0.0)
        reasons["preferred_level"] = f"NLQF niveau {'match' if match else 'geen match'}: gewenst={profile.preferred_level}, module={row[level_col]}."
    else:
        reasons["preferred_level"] = "Geen NLQF niveau-voorkeur opgegeven."

    ## 4. Max Difficulty Constraint (Behouden)
    diff_col = COLS.get("estimated_difficulty")
    if profile.max_difficulty is not None and diff_col and diff_col in row.index:
        diff_val = extract_numeric_series(pd.Series([row[diff_col]])).iloc[0]
        
        if pd.isna(diff_val):
            scores.append(0.5)
            reasons["difficulty"] = "Moeilijkheid onbekend; neutrale score."
        else:
            # De module mag niet moeilijker zijn dan het maximum van de student
            match = diff_val <= profile.max_difficulty
            scores.append(1.0 if match else 0.0)
            reasons["difficulty"] = f"Moeilijkheid {'binnen' if match else 'boven'} voorkeursniveau (≈ {diff_val})."
    else:
        reasons["difficulty"] = "Geen moeilijkheids-constraint of kolom."


    if not scores:
        return 0.0, reasons

    # De eindscore is het gemiddelde van alle actieve constraint scores
    return float(np.mean(scores)), reasons


In [10]:
def recommend(
    profile: StudentProfile,
    k: int = 10,
    alpha: float = 0.7,   # gewicht content
    beta: float = 0.2,    # gewicht constraints
    gamma: float = 0.1    # gewicht populariteit
) -> pd.DataFrame:
    if not np.isclose(alpha + beta + gamma, 1.0):
        raise ValueError("alpha + beta + gamma moeten samen 1.0 zijn.")

    p_vec = profile_vector(profile)
    content_scores = cosine_similarity(p_vec, X).flatten()

    # Populariteit (optioneel)
    pop_col = COLS.get("popularity_score")
    if pop_col and pop_col in df.columns:
        pop_raw = extract_numeric_series(df[pop_col]).fillna(0.0)
        pop_scaled = (pop_raw - pop_raw.min()) / (pop_raw.max() - pop_raw.min() + 1e-9)
    else:
        pop_scaled = pd.Series(0.0, index=df.index)

    rows = []
    title_col = COLS.get("title") or df.columns[0]
    loc_col = COLS.get("location")

    for idx, row in df.iterrows():
        cstr_score, reasons = constraint_score(row, profile)
        c_score = float(content_scores[idx])
        pop_score = float(pop_scaled.loc[idx])

        final_score = alpha * c_score + beta * cstr_score + gamma * pop_score

        rows.append({
            "index": idx,
            "module_title": row.get(title_col, ""),
            "location": row.get(loc_col, ""),
            "final_score": final_score,
            "content_sim": c_score,
            "constraint_score": cstr_score,
            "popularity_score": pop_score,
            "constraint_reasons": reasons,
        })

    rec_df = pd.DataFrame(rows).sort_values("final_score", ascending=False).head(k).reset_index(drop=True)
    return rec_df


In [11]:
def diversity_score(indices: List[int]) -> float:
    if len(indices) < 2:
        return 0.0
    sub = X[indices]
    sim = cosine_similarity(sub)
    n = sim.shape[0]
    mask = np.triu(np.ones_like(sim, dtype=bool), k=1)
    if mask.sum() == 0:
        return 0.0
    avg_sim = sim[mask].mean()
    return float(1.0 - avg_sim)

def explain_overlap(profile_text: str, module_text: str, top_k: int = 8) -> List[str]:
    def tokenize(s: str) -> List[str]:
        return re.findall(r"\w+", s.lower())

    p_tokens = tokenize(profile_text)
    m_tokens = tokenize(module_text)

    common = [t for t in m_tokens if t in set(p_tokens)]
    counts: Dict[str, int] = {}
    for t in common:
        counts[t] = counts.get(t, 0) + 1

    sorted_terms = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return [w for w, _ in sorted_terms[:top_k]]


In [13]:
demo_profile = StudentProfile(
    interests_text="artificial intelligence data science machine learning programmeren analyse van data",
)

demo_recs = recommend(demo_profile, k=5, alpha=0.7, beta=0.2, gamma=0.1)
demo_recs


,index,module_title,location,final_score,content_sim,constraint_score,popularity_score,constraint_reasons
0,146,Smart Industry & Internet of Things,Breda,0.196629,0.225797,0.0,0.385714,"{'location': 'Geen locatie-voorkeur opgegeven.', 'study_credits': 'Geen studiepunten-constraint of kolom.', 'preferred_level': 'Geen NLQF niveau-voorkeur opgegeven.', 'difficulty': 'Geen moeilijkh..."
1,167,Forensische Chemie,Breda en Den Bosch,0.185206,0.234842,0.0,0.208163,"{'location': 'Geen locatie-voorkeur opgegeven.', 'study_credits': 'Geen studiepunten-constraint of kolom.', 'preferred_level': 'Geen NLQF niveau-voorkeur opgegeven.', 'difficulty': 'Geen moeilijkh..."
2,171,Molecular Modeling & Data-driven Analysis,Breda,0.168327,0.100817,0.0,0.977551,"{'location': 'Geen locatie-voorkeur opgegeven.', 'study_credits': 'Geen studiepunten-constraint of kolom.', 'preferred_level': 'Geen NLQF niveau-voorkeur opgegeven.', 'difficulty': 'Geen moeilijkh..."
3,131,AI Driven Robotics,Breda,0.135507,0.059762,0.0,0.936735,"{'location': 'Geen locatie-voorkeur opgegeven.', 'study_credits': 'Geen studiepunten-constraint of kolom.', 'preferred_level': 'Geen NLQF niveau-voorkeur opgegeven.', 'difficulty': 'Geen moeilijkh..."
4,130,Datagedreven besluitvorming met AI,Breda,0.132836,0.148658,0.0,0.287755,"{'location': 'Geen locatie-voorkeur opgegeven.', 'study_credits': 'Geen studiepunten-constraint of kolom.', 'preferred_level': 'Geen NLQF niveau-voorkeur opgegeven.', 'difficulty': 'Geen moeilijkh..."


In [14]:
# Diversiteitsscore van de aanbevelingen
indices = demo_recs["index"].tolist()
print("Diversiteitsscore (0 = zeer gelijkend, 1 = zeer divers):", diversity_score(indices))

# Toon overlappende termen voor de eerste 3 aanbevelingen
title_col = "module_title"
for _, row in demo_recs.head(3).iterrows():
    idx = row["index"]
    module_text = df.loc[idx, "module_text"]
    overlap = explain_overlap(demo_profile.interests_text, module_text, top_k=8)
    print(f"\nModule: {row[title_col]}")
    print("Overlappende trefwoorden:", ", ".join(overlap) if overlap else "(geen expliciete overlap gevonden)")


Diversiteitsscore (0 = zeer gelijkend, 1 = zeer divers): 0.9718899387841465

Module: Smart Industry & Internet of Things
Overlappende trefwoorden: data, artificial, intelligence, machine, learning, van

Module: Forensische Chemie
Overlappende trefwoorden: analyse, van, data, science

Module: Molecular Modeling & Data-driven Analysis
Overlappende trefwoorden: data, programmeren, machine, learning
